In [1]:
import pandas as pd
import numpy as np

# Cargar archivos
df_clean = pd.read_csv("../data/processed/gaid_clean.csv")
selected_metrics = pd.read_csv("../data/processed/selected_metrics.csv")
dim_country = pd.read_csv("../data/processed/dim_country_enriched.csv")

# Filtrar métricas seleccionadas
df_selected = df_clean.merge(
    selected_metrics,
    on="Metric",
    how="inner"
)

# Tomar último año disponible por país y métrica
df_latest = (
    df_selected
    .sort_values(["Country", "ISO3", "Metric", "Year"])
    .groupby(["Country", "ISO3", "Metric"], as_index=False)
    .tail(1)
    .copy()
)

# Normalización por métrica
def normalize_metric(x):
    if x.max() == x.min():
        return pd.Series([100] * len(x), index=x.index)
    return ((x - x.min()) / (x.max() - x.min())) * 100

df_latest["NormalizedValue"] = df_latest.groupby("Metric")["Value"].transform(normalize_metric)

# Aplicar peso
df_latest["WeightedValue"] = df_latest["NormalizedValue"] * df_latest["Weight"]

# Excluir pilar General del índice propio
df_index = df_latest[df_latest["Pillar"] != "General"].copy()

# Calcular score por pilar
pillar_scores = (
    df_index
    .groupby(["Country", "ISO3", "Pillar"])
    .apply(
        lambda g: pd.Series({
            "PillarScore": g["WeightedValue"].sum() / g["Weight"].sum(),
            "MetricCount": g["Metric"].nunique()
        })
    )
    .reset_index()
)

# Pasar a formato ancho
ai_scores = pillar_scores.pivot_table(
    index=["Country", "ISO3"],
    columns="Pillar",
    values="PillarScore",
    aggfunc="mean"
).reset_index()

# Asegurar columnas de pilares
pillar_cols = ["Economy", "Governance", "Infrastructure", "Innovation", "Talent"]

for col in pillar_cols:
    if col not in ai_scores.columns:
        ai_scores[col] = np.nan

# Cobertura de pilares
ai_scores["PillarCoverage"] = ai_scores[pillar_cols].notna().sum(axis=1)

# Score usando todos los pilares disponibles
ai_scores["AI_Readiness_Score_AllAvailable"] = ai_scores[pillar_cols].mean(axis=1)

# Clasificación de calidad
ai_scores["Score_Quality"] = np.select(
    [
        ai_scores["PillarCoverage"] == 5,
        ai_scores["PillarCoverage"] >= 4
    ],
    [
        "Full coverage",
        "Comparable partial"
    ],
    default="Low coverage"
)

# Score comparable: solo países con 4 o 5 pilares
ai_scores["AI_Readiness_Score_Comparable"] = np.where(
    ai_scores["PillarCoverage"] >= 4,
    ai_scores["AI_Readiness_Score_AllAvailable"],
    np.nan
)

# Ranking comparable
ai_scores["Global_Rank_Comparable"] = ai_scores["AI_Readiness_Score_Comparable"].rank(
    ascending=False,
    method="dense"
)

# Unir con dimensión país enriquecida
ai_scores_model = ai_scores.merge(
    dim_country[["CountryKey", "Country", "ISO3", "Region", "IsLatAm", "IsPeru"]],
    on=["Country", "ISO3"],
    how="left"
)

# Ordenar columnas
final_cols = [
    "CountryKey",
    "Country",
    "ISO3",
    "Region",
    "IsLatAm",
    "IsPeru",
    "Economy",
    "Governance",
    "Infrastructure",
    "Innovation",
    "Talent",
    "PillarCoverage",
    "Score_Quality",
    "AI_Readiness_Score_AllAvailable",
    "AI_Readiness_Score_Comparable",
    "Global_Rank_Comparable"
]

ai_scores_model = ai_scores_model[final_cols]

# Exportar
ai_scores_model.to_csv("../data/processed/ai_readiness_scores_model_final.csv", index=False)

ai_scores_model.head()

,CountryKey,Country,ISO3,Region,IsLatAm,IsPeru,Economy,Governance,Infrastructure,Innovation,Talent,PillarCoverage,Score_Quality,AI_Readiness_Score_AllAvailable,AI_Readiness_Score_Comparable,Global_Rank_Comparable
0,1,Afghanistan,AFG,Other Regions,False,False,NaN,3.981716,0.527215,0.003581,NaN,3,Low coverage,1.504171,NaN,NaN
1,2,Albania,ALB,Other Regions,False,False,NaN,39.502273,93.245628,0.010743,38.144330,4,Comparable partial,42.725744,42.725744,13.0
2,3,Algeria,DZA,Other Regions,False,False,0.0,19.314128,36.232427,1.212623,35.374570,5,Full coverage,18.426750,18.426750,111.0
3,4,Andorra,AND,Other Regions,False,False,NaN,38.469275,45.346041,0.003581,54.123711,4,Comparable partial,34.485652,34.485652,41.0
4,5,Angola,AGO,Other Regions,False,False,NaN,59.013930,57.156299,0.007162,NaN,3,Low coverage,38.725797,NaN,NaN
